<a href="https://colab.research.google.com/github/AliMehdii/Memoire-2023/blob/master/Code/Version_01_Modeling_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import os
import numpy as np
import pandas as pd
import cv2

# import splitfolders
import h5py
from matplotlib import pyplot as plt
%matplotlib inline
from matplotlib import rcParams
import seaborn as sns
from PIL import Image
import imutils 

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.applications import (
    InceptionResNetV2,
    ResNet50,
    InceptionV3,
    DenseNet121,
)
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.imagenet_utils import preprocess_input



In [20]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [21]:
try:
    from google.colab import drive
    drive.mount("/content/drive/", force_remount=True)
    google_drive_prefix = "/content/drive/My Drive"
    data_prefix = "{}/mnist/".format(google_drive_prefix)
except ModuleNotFoundError: 
    data_prefix = "data/"

Mounted at /content/drive/


In [22]:
train_set = '/content/drive/My Drive/Cropped_Image_Sets/train'
val_set = '/content/drive/My Drive/Cropped_Image_Sets/val'
test_set = '/content/drive/My Drive/Cropped_Image_Sets/test'
model_dir ="/content/drive/My Drive/Models/RadImageNet-ResNet50_notop.h5"
IMAGE_SIZE = 128

In [23]:
def init_data(train_dir: str, valid_dir: str) -> tuple:
    train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1/255.0
    )
    valid_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1/255.0
    )
    
    train_data = train_datagen.flow_from_directory(
        directory=train_dir,
        class_mode='categorical',
        target_size=(128, 128),
        batch_size=32,
        seed=42,
        shuffle=False,
    )
    valid_data = valid_datagen.flow_from_directory(
        directory=valid_dir,
        class_mode='categorical',
        target_size=(128, 128),
        batch_size=32,
        seed=42,
        shuffle=False,
    )
    
    return train_data, valid_data

In [24]:
train_data, valid_data = init_data(train_dir=train_set, valid_dir=val_set)

Found 2144 images belonging to 3 classes.
Found 458 images belonging to 3 classes.


In [25]:
def build_transfer_learning_model(base_model):
    # `base_model` stands for the pretrained model
    # We want to use the learned weights, and to do so we must freeze them
    for layer in base_model.layers:
        layer.trainable = False
        
    # Declare a sequential model that combines the base model with custom layers
    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=3, activation='softmax')
    ])
    
    # Compile the model
    model.compile(
        loss='categorical_crossentropy',
        optimizer='Adam',
        metrics=['accuracy']
    )
    
    return model

In [26]:
rad_model = build_transfer_learning_model(
    base_model=ResNet50(weights= model_dir, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),  include_top=False, pooling="avg")
)

In [15]:
rad_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg16 (Functional)          (None, 8, 8, 512)         14714688  
                                                                 
 global_average_pooling2d_1   (None, 512)              0         
 (GlobalAveragePooling2D)                                        
                                                                 
 batch_normalization_1 (Batc  (None, 512)              2048      
 hNormalization)                                                 
                                                                 
 dropout_1 (Dropout)         (None, 512)               0         
                                                                 
 dense_1 (Dense)             (None, 3)                 1539      
                                                                 
Total params: 14,718,275
Trainable params: 2,563
Non-t

In [16]:
# Train the model for 10 epochs
rad_hist = rad_model.fit(
    train_data,
    validation_data=valid_data,
    epochs=10
)

Epoch 1/10
67/67 [==============================] - 283s 4s/step - loss: 1.2800 - accuracy: 0.3368 - val_loss: 1.1372 - val_accuracy: 0.3362
Epoch 2/10
67/67 [==============================] - 9s 130ms/step - loss: 1.1763 - accuracy: 0.3890 - val_loss: 1.0943 - val_accuracy: 0.4629
Epoch 3/10
67/67 [==============================] - 10s 145ms/step - loss: 1.1426 - accuracy: 0.3974 - val_loss: 1.0691 - val_accuracy: 0.4651
Epoch 4/10
67/67 [==============================] - 10s 145ms/step - loss: 1.1240 - accuracy: 0.4002 - val_loss: 1.0629 - val_accuracy: 0.4672
Epoch 5/10
67/67 [==============================] - 9s 129ms/step - loss: 1.1172 - accuracy: 0.4016 - val_loss: 1.0679 - val_accuracy: 0.4389
Epoch 6/10
67/67 [==============================] - 9s 141ms/step - loss: 1.0964 - accuracy: 0.4249 - val_loss: 1.0735 - val_accuracy: 0.4367
Epoch 7/10
67/67 [==============================] - 9s 141ms/step - loss: 1.0993 - accuracy: 0.4286 - val_loss: 1.0595 - val_accuracy: 0.4651
Epoch

In [ ]:
# IMAGE_SIZE = 256

# model_dir ="C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Brain MRI images Classification/Models/RadImageNet-ResNet50_notop.h5"
# # base_model = ResNet50(weights= model_dir, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False, pooling="avg")
# base_model = ResNet50(weights= model_dir, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),  include_top=False, pooling="avg")
# base_model.summary()

Model: "resnet50"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_10 (InputLayer)          [(None, 256, 256, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv1_pad (ZeroPadding2D)      (None, 262, 262, 3)  0           ['input_10[0][0]']               
                                                                                                  
 conv1_conv (Conv2D)            (None, 128, 128, 64  9472        ['conv1_pad[0][0]']              
                                )                                                                 
                                                                                           

In [ ]:
# base_model_config = base_model.get_config() 

In [ ]:
# channel_num = 1
# base_model_config["layers"][0]["config"]["batch_input_shape"] =(None, IMAGE_SIZE, IMAGE_SIZE, channel_num)


In [ ]:
# updated_base_model = Model.from_config(base_model_config)
# print(updated_base_model.summary())

Model: "resnet50"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_9 (InputLayer)           [(None, 256, 256, 1  0           []                               
                                )]                                                                
                                                                                                  
 conv1_pad (ZeroPadding2D)      (None, 262, 262, 1)  0           ['input_9[0][0]']                
                                                                                                  
 conv1_conv (Conv2D)            (None, 128, 128, 64  3200        ['conv1_pad[0][0]']              
                                )                                                                 
                                                                                           

In [ ]:

# data = []
# Home = 'C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train'
# for folder in sorted(os.listdir('C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train')):
#     for file in sorted(os.listdir('C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train/'+folder)):
#         file_path = Home + '/' + folder + '/' + file

#         file_name = file.split('.')[0]
#         data.append([file_name, folder, file_path])

# train_set = pd.DataFrame(data, columns=['File_Name', 'Folder', 'File_Path'])
# print(train_set)
# train_set.to_csv('Train_set.csv')
